In [ ]:
import time
from itertools import product

import numpy as np
import pandas as pd
from file import LSM, Disk


# ============================================================
# Workload Generators
# ============================================================
def workload_uniform(n_ops, key_range):
    for _ in range(n_ops):
        op = np.random.random()
        k = np.random.randint(0, key_range)
        if op < 0.75:
            yield ("put", k, np.random.randint(1, 1000))
        elif op < 0.90:
            yield ("delete", k)
        else:
            yield ("get", k)


def workload_zipf(n_ops, key_range, theta=1.2):
    keys = np.random.zipf(theta, size=n_ops) % key_range
    for k in keys:
        if np.random.random() < 0.8:
            yield ("put", int(k), np.random.randint(1, 1000))
        else:
            yield ("get", int(k))


def workload_skewed(n_ops, key_range, hot_ratio=0.1):
    hot_keys = int(key_range * hot_ratio)
    for _ in range(n_ops):
        if np.random.random() < 0.8:
            k = np.random.randint(0, hot_keys)
        else:
            k = np.random.randint(hot_keys, key_range)
        if np.random.random() < 0.7:
            yield ("put", k, np.random.randint(1, 1000))
        else:
            yield ("get", k)


# ============================================================
# Metrics extractors
# ============================================================
def space_amplification(lsm, logical_keys):
    physical = 0
    for level in lsm.levels:
        for fid in level:
            f = lsm.disk.files.get(fid)
            if f and f.values:
                physical += len(f.values)
    return physical / logical_keys


def write_amplification(lsm, mem_flushes):
    return lsm.disk.write_ops - mem_flushes


def read_amplification(lsm):
    return lsm.disk.read_ops


def read_write_latency(latency_records):
    """Return mean read latency, mean write latency."""
    reads = [t for kind, t in latency_records if kind == "read"]
    writes = [t for kind, t in latency_records if kind == "write"]

    read_lat = np.mean(reads) if reads else 0
    write_lat = np.mean(writes) if writes else 0
    return read_lat, write_lat


# ============================================================
# Run single experiment configuration
# ============================================================
def run_single_experiment(
    compaction_mode,
    memtable_size,
    l0_files,
    level_mult,
    workload_type,
    n_ops,
    key_range,
):
    disk = Disk()
    lsm = LSM(
        disk=disk,
        mem_capacity=memtable_size,
        sst_capacity=memtable_size,
        max_l0_files=l0_files,
        level_multiplier=level_mult,
        compaction_mode=compaction_mode,
    )

    # Pick workload
    if workload_type == "uniform":
        ops = workload_uniform(n_ops, key_range)
    elif workload_type == "zipf":
        ops = workload_zipf(n_ops, key_range)
    else:
        ops = workload_skewed(n_ops, key_range)

    mem_flushes = 0
    # latency_records = []

    for op in ops:
        if op[0] == "put":
            _, key, v = op
            t0 = time.time()
            lsm.put(key, v)
            t1 = time.time()
            # latency_records.append(("write", t1 - t0))
            if len(lsm.mem) == 0:
                mem_flushes += 1

        elif op[0] == "delete":
            _, key = op
            t0 = time.time()
            lsm.delete(key)
            t1 = time.time()
            # latency_records.append(("write", t1 - t0))
            if len(lsm.mem) == 0:
                mem_flushes += 1

        else:  # get
            _, key = op
            t0 = time.time()
            lsm.get(key)
            t1 = time.time()
            # latency_records.append(("read", t1 - t0))

    # Compute metrics
    # read_lat, write_lat = read_write_latency(latency_records)
    wa = write_amplification(lsm, mem_flushes)
    ra = read_amplification(lsm)
    sa = space_amplification(lsm, key_range)

    return {
        "compaction": compaction_mode,
        "memtable": memtable_size,
        "l0": l0_files,
        "level_mult": level_mult,
        "workload": workload_type,
        "ops": n_ops,
        "key_range": key_range,
        "write_amp": wa,
        "read_amp": ra,
        "space_amp": sa,
        # "read_latency": read_lat,
        # "write_latency": write_lat,
    }


# ============================================================
# Full Factorial Experiment Runner
# ============================================================
def run_experiments(
    compaction_modes=["tiered", "leveled", "fifo", "hybrid"],
    memtable_sizes=[1, 2, 4, 8, 16],
    l0_sizes=[4, 8, 12],
    level_mults=[5, 10, 100],
    workloads=["uniform", "zipf", "skewed"],
    n_ops=50_000,
    key_range=10_000,
):
    results = []

    configs = product(
        compaction_modes,
        memtable_sizes,
        l0_sizes,
        level_mults,
        workloads,
    )

    for compaction, mem, l0, mult, wl in configs:
        print(f"▶ Running {compaction}, mem={mem}, L0={l0}, M={mult}, WL={wl}")
        res = run_single_experiment(
            compaction,
            mem,
            l0,
            mult,
            wl,
            n_ops,
            key_range,
        )
        results.append(res)

    return pd.DataFrame(results)


# ============================================================
# Example execution
# ============================================================
# df = run_experiments(
#     compaction_modes=["tiered", "leveled"],   # you can expand this
#     memtable_sizes=[4, 8],                   # small search for demo
#     l0_sizes=[4, 8],
#     level_mults=[5, 10],
#     workloads=["uniform", "zipf"],
#     n_ops=20_000,
#     key_range=5_000,
# )
# df = run_experiments(
#     compaction_modes=["tiered", "leveled"],   # you can expand this
#     memtable_sizes=[4],                   # small search for demo
#     l0_sizes=[4],
#     level_mults=[5],
#     workloads=["uniform"],
#     n_ops=200_000,
#     key_range=5_000,
# )

# print(df)


1. compaction strategy

In [9]:
df = run_experiments(
    compaction_modes=["tiered", "leveled"],   # you can expand this
    memtable_sizes=[1000],                   # small search for demo
    l0_sizes=[4],
    level_mults=[4],
    workloads=["uniform"],
    n_ops=200_000,
    key_range=5_000,
)
df
#tiered compaction is better for write amplification
#leveled compaction is better for space amplification

▶ Running tiered, mem=1000, L0=4, M=4, WL=uniform
▶ Running leveled, mem=1000, L0=4, M=4, WL=uniform


,compaction,memtable,l0,level_mult,workload,ops,key_range,write_amp,read_amp,space_amp
0,tiered,1000,4,4,uniform,200000,5000,179,84664,7.5844
1,leveled,1000,4,4,uniform,200000,5000,861,80238,1.8000


2. ssttable Size (=sst_capacity)

In [12]:
df = run_experiments(
    compaction_modes=["tiered", "leveled"],   # you can expand this
    memtable_sizes=[1000, 2000, 4000, 8000, 16000],                   # small search for demo
    l0_sizes=[4],
    level_mults=[4],
    workloads=["uniform"],
    n_ops=200_000,
    key_range=5_000,
)
df

▶ Running tiered, mem=1000, L0=4, M=4, WL=uniform
▶ Running tiered, mem=2000, L0=4, M=4, WL=uniform
▶ Running tiered, mem=4000, L0=4, M=4, WL=uniform
▶ Running tiered, mem=8000, L0=4, M=4, WL=uniform
▶ Running tiered, mem=16000, L0=4, M=4, WL=uniform
▶ Running leveled, mem=1000, L0=4, M=4, WL=uniform
▶ Running leveled, mem=2000, L0=4, M=4, WL=uniform
▶ Running leveled, mem=4000, L0=4, M=4, WL=uniform
▶ Running leveled, mem=8000, L0=4, M=4, WL=uniform
▶ Running leveled, mem=16000, L0=4, M=4, WL=uniform


,compaction,memtable,l0,level_mult,workload,ops,key_range,write_amp,read_amp,space_amp
0,tiered,1000,4,4,uniform,200000,5000,175,86389,7.7586
1,tiered,2000,4,4,uniform,200000,5000,63,46468,3.0000
2,tiered,4000,4,4,uniform,200000,5000,16,24260,11.0412
3,tiered,8000,4,4,uniform,200000,5000,4,10935,7.1986
4,tiered,16000,4,4,uniform,200000,5000,2,5884,5.2000
5,leveled,1000,4,4,uniform,200000,5000,865,80521,1.8000
6,leveled,2000,4,4,uniform,200000,5000,250,46869,2.6000
7,leveled,4000,4,4,uniform,200000,5000,81,24205,4.2000
8,leveled,8000,4,4,uniform,200000,5000,18,12030,7.4000
9,leveled,16000,4,4,uniform,200000,5000,7,5820,13.8000


3.Level 0 size

In [13]:
df = run_experiments(
    compaction_modes=["tiered", "leveled"],   # you can expand this
    memtable_sizes=[1000],                   # small search for demo
    l0_sizes=[4,8,12],
    level_mults=[4],
    workloads=["uniform"],
    n_ops=200_000,
    key_range=5_000,
)
df

▶ Running tiered, mem=1000, L0=4, M=4, WL=uniform
▶ Running tiered, mem=1000, L0=8, M=4, WL=uniform
▶ Running tiered, mem=1000, L0=12, M=4, WL=uniform
▶ Running leveled, mem=1000, L0=4, M=4, WL=uniform
▶ Running leveled, mem=1000, L0=8, M=4, WL=uniform
▶ Running leveled, mem=1000, L0=12, M=4, WL=uniform


,compaction,memtable,l0,level_mult,workload,ops,key_range,write_amp,read_amp,space_amp
0,tiered,1000,4,4,uniform,200000,5000,179,86086,7.5856
1,tiered,1000,8,4,uniform,200000,5000,110,86562,7.0278
2,tiered,1000,12,4,uniform,200000,5000,70,85052,5.7824
3,leveled,1000,4,4,uniform,200000,5000,865,79499,1.8000
4,leveled,1000,8,4,uniform,200000,5000,840,90072,2.6000
5,leveled,1000,12,4,uniform,200000,5000,821,93826,3.4000


4. Level Multiplier

In [14]:
df = run_experiments(
    compaction_modes=["tiered", "leveled"],   # you can expand this
    memtable_sizes=[1000],                   # small search for demo
    l0_sizes=[4],
    level_mults=[4, 8, 16, 32, 64, 128],
    workloads=["uniform"],
    n_ops=200_000,
    key_range=5_000,
)
df

▶ Running tiered, mem=1000, L0=4, M=4, WL=uniform
▶ Running tiered, mem=1000, L0=4, M=8, WL=uniform
▶ Running tiered, mem=1000, L0=4, M=16, WL=uniform
▶ Running tiered, mem=1000, L0=4, M=32, WL=uniform
▶ Running tiered, mem=1000, L0=4, M=64, WL=uniform
▶ Running tiered, mem=1000, L0=4, M=128, WL=uniform
▶ Running leveled, mem=1000, L0=4, M=4, WL=uniform
▶ Running leveled, mem=1000, L0=4, M=8, WL=uniform
▶ Running leveled, mem=1000, L0=4, M=16, WL=uniform
▶ Running leveled, mem=1000, L0=4, M=32, WL=uniform
▶ Running leveled, mem=1000, L0=4, M=64, WL=uniform
▶ Running leveled, mem=1000, L0=4, M=128, WL=uniform


,compaction,memtable,l0,level_mult,workload,ops,key_range,write_amp,read_amp,space_amp
0,tiered,1000,4,4,uniform,200000,5000,179,84898,7.5720
1,tiered,1000,4,8,uniform,200000,5000,164,89867,3.9992
2,tiered,1000,4,16,uniform,200000,5000,154,90262,3.2562
3,tiered,1000,4,32,uniform,200000,5000,149,92066,2.8936
4,tiered,1000,4,64,uniform,200000,5000,140,95437,22.9378
5,tiered,1000,4,128,uniform,200000,5000,144,93651,22.7460
6,leveled,1000,4,4,uniform,200000,5000,866,79666,1.8000
7,leveled,1000,4,8,uniform,200000,5000,861,80599,1.8000
8,leveled,1000,4,16,uniform,200000,5000,866,79146,1.8000
9,leveled,1000,4,32,uniform,200000,5000,865,79362,1.8000


5. Workload

In [15]:
df = run_experiments(
    compaction_modes=["tiered", "leveled"],   # you can expand this
    memtable_sizes=[1000],                   # small search for demo
    l0_sizes=[4],
    level_mults=[4],
    workloads=["uniform", "zipf", "skewed"],
    n_ops=200_000,
    key_range=5_000,
)
df

▶ Running tiered, mem=1000, L0=4, M=4, WL=uniform
▶ Running tiered, mem=1000, L0=4, M=4, WL=zipf
▶ Running tiered, mem=1000, L0=4, M=4, WL=skewed
▶ Running leveled, mem=1000, L0=4, M=4, WL=uniform
▶ Running leveled, mem=1000, L0=4, M=4, WL=zipf
▶ Running leveled, mem=1000, L0=4, M=4, WL=skewed


,compaction,memtable,l0,level_mult,workload,ops,key_range,write_amp,read_amp,space_amp
0,tiered,1000,4,4,uniform,200000,5000,175,85115,7.7538
1,tiered,1000,4,4,zipf,200000,5000,79,80263,4.1172
2,tiered,1000,4,4,skewed,200000,5000,69,124470,3.4334
3,leveled,1000,4,4,uniform,200000,5000,865,79462,1.8000
4,leveled,1000,4,4,zipf,200000,5000,723,76514,1.7988
5,leveled,1000,4,4,skewed,200000,5000,617,102087,1.7970


In [11]:
df.sort_values(by=['compaction', 'workload'])

,compaction,memtable,l0,level_mult,workload,ops,key_range,write_amp,read_amp,space_amp
5,leveled,1000,4,4,uniform,200000,5000,860,80524,1.8000
6,leveled,2000,4,4,uniform,200000,5000,250,46388,2.6000
7,leveled,4000,4,4,uniform,200000,5000,79,23764,4.2000
8,leveled,8000,4,4,uniform,200000,5000,18,11841,7.4000
9,leveled,16000,4,4,uniform,200000,5000,7,5888,13.8000
0,tiered,1000,4,4,uniform,200000,5000,175,85215,7.7524
1,tiered,2000,4,4,uniform,200000,5000,63,46393,3.0000
2,tiered,4000,4,4,uniform,200000,5000,20,24280,1.0000
3,tiered,8000,4,4,uniform,200000,5000,4,11419,7.1980
4,tiered,16000,4,4,uniform,200000,5000,2,5889,5.2000
